In [4]:
import sys
import torch
import wandb
import torch.nn.functional as F

from neuralop import H1Loss, LpLoss, BurgersEqnLoss, ICLoss, get_model
from neuralop.data.datasets import load_mini_burgers_1dtime
from neuralop.training import AdamW
from neuralop.utils import get_wandb_api_key, count_model_params, get_project_root
from neuralop.losses.meta_losses import Relobralo, SoftAdapt

# Read the configuration
config_name = "default"
from zencfg import make_config_from_cli
import sys

#sys.path.insert(0, "../")
from default_config import Default

device = "cuda" #GPU

In [5]:
config = make_config_from_cli(Default)
config = config.to_dict()


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Set up WandB logging
if config.wandb.log:
    wandb.login(key=get_wandb_api_key())
    if config.wandb.name:
        wandb_name = config.wandb.name
    else:
        wandb_name = "_".join(
            f"{var}"
            for var in [
                config_name,
                config.model.model_arch,
                config.model.n_layers,
                config.model.n_modes,
                config.model.hidden_channels,
            ]
        )
    wandb_init_args = dict(
        config=config,
        name=wandb_name,
        group=config.wandb.group,
        project=config.wandb.project,
        entity=config.wandb.entity,
    )
    if config.wandb.sweep:
        for key in wandb.config.keys():
            config.params[key] = wandb.config[key]
    wandb.init(**wandb_init_args)
else:
    wandb_init_args = None


# Print configuration details
if config.verbose:
    print("##### CONFIG ######")
    print(config)
    sys.stdout.flush()

# Data loading
data_path = get_project_root() / config.data.folder
# Load the Burgers dataset with specified parameters
train_loader, test_loaders, data_processor = load_mini_burgers_1dtime(
    data_path=data_path,
    n_train=config.data.n_train,
    batch_size=config.data.batch_size,
    n_test=config.data.n_tests[0],
    test_batch_size=config.data.test_batch_sizes[0],
    temporal_subsample=config.data.get("temporal_subsample", 1),
    spatial_subsample=config.data.get("spatial_subsample", 1),
)

# Create the model
model = get_model(config)
model = model.to(device)

ValueError: Arguments must be in pairs like: --model._config_name MyModel --model.layers 24